# Activation Oracle explanations — Kaggle, multi-layer sweep

Adapted from `explanation.ipynb` (the reference notebook your supervisor ran).

**What changed vs. the reference notebook, and why:**

- **Environment**: runs on Kaggle — package installs, `kaggle_secrets` HF login, `device_map="auto"`
  loading, paths under `/kaggle/input/...` and `/kaggle/working/...` (taken from your
  `oracle-explainability` notebook).
- **Layer transfer**: the reference notebook hardcodes `LAYER = 18` and only ever probes one
  source layer. Your notebook sweeps `LAYERS = [9, 18, 27]` and reports the source layer inside
  every result — that sweep is restored here for every probe (A01–A08, A16, A09, A13, A15).
- **Documents**: generalized from the reference's single doc `"1"` (and the 3-doc slice in the
  A16/A09/A13/A15 cells) to your five dataset indices `DOC_IDS = [193, 198, 27, 61, 75]`.
- **Injection / hook math is left untouched.** `get_introspection_prefix`, `find_placeholders`,
  `get_injection_submodule` (fixed at oracle block 1 — the source layer is *never* used to pick
  the oracle block, only quoted in the prompt text), `make_injection_hook` (Eq. 1: norm-matched
  additive injection), and `run_batch`'s batching/left-padding are copied verbatim from the
  reference notebook. This is the part you confirmed is correct — nothing here deviates from it.
- **Cleaner prints**: every probe now prints one line per generation as it runs, plus a compact
  grouped summary at the end, instead of dumping everything only after the whole sweep finishes.
- All five docs × all three layers × every probe are saved to `oracle_outputs/*.json` as you go,
  so a crash partway through doesn't lose earlier results.

**Before running:** set `ACTIVATIONS_DIR` in the config cell to your dataset's actual path
(e.g. `/kaggle/input/<your-dataset-slug>`), and set `HF_TOKEN` in Kaggle secrets. The loader
below searches recursively for files named `{doc}_{status}_L{layer}_{element}.pt` and
`..._a16.pt`, so it works whether your dataset has `activations_steered/` /
`activations_unsteered/` subfolders or everything flat in one directory — but it assumes that
naming convention. If your five files are named differently, adjust `FILENAME_PATTERN` in the
config cell.

## 1. Install packages

In [ ]:
!pip install -q --upgrade \
    "bitsandbytes>=0.48.1"
!pip install -q "torchao>0.16.0"


## 2. Imports, versions, HF login

In [ ]:
import os, gc, json, time, contextlib
from pathlib import Path
from typing import Dict, List, Optional
from collections import defaultdict

import torch
import torch.nn.functional as F
import transformers

torch.set_grad_enabled(False)

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)

print(f"transformers {transformers.__version__} | torch {torch.__version__}")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## 3. Configuration

**Edit `ACTIVATIONS_DIR` before running.**

In [ ]:
# --- Paths ---
ACTIVATIONS_DIR = "/kaggle/input/<your-dataset-slug>"   # <-- EDIT THIS
OUTPUT_DIR       = "/kaggle/working/oracle_outputs"

# Filename convention the loaders below expect (searched recursively via rglob):
#   {doc}_{status}_L{layer}_{element}.pt          e.g. 193_unsteered_L18_p.pt
#   {doc}_{status}_L{layer}_{element}_a16.pt       (A16 sidecar)
FILENAME_PATTERN = "{doc}_{status}_L{layer}_{element}.pt"
SIDECAR_PATTERN  = "{doc}_{status}_L{layer}_{element}_a16.pt"

# --- Documents & layers (the "layer transfer" sweep) ---
DOC_IDS = [193, 198, 27, 61, 75]
LAYERS  = [9, 18, 27]
ELEMENTS = ["p", "i", "c", "o"]

# --- Models ---
BASE_MODEL     = "Qwen/Qwen3-8B"
ORACLE_ADAPTER = "adamkarvonen/checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B"

# --- Injection math (verbatim from the reference notebook; do not change) ---
INJECTION_LAYER = 1
STEERING_COEFFICIENT = 1.0
MAX_NEW_TOKENS = 60
BATCH_SIZE = 64

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Docs: {DOC_IDS}")
print(f"Layers (sweep): {LAYERS}")
print(f"Activations dir: {ACTIVATIONS_DIR}")


## 4. Load Qwen3-8B base + the pretrained Activation Oracle LoRA

Kaggle-style loading (`device_map="auto"`), otherwise identical to the reference notebook.

In [ ]:
from peft import LoraConfig, PeftConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

_sid = tokenizer.encode(" ?", add_special_tokens=False)
assert len(_sid) == 1, f"' ?' is not a single token: {_sid}"

_maj, _min = (int(x) for x in transformers.__version__.split(".")[:2])
_dtype_kw = "dtype" if (_maj, _min) >= (4, 56) else "torch_dtype"

print("Loading base model (bfloat16, device_map=auto)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", **{_dtype_kw: torch.bfloat16}
)
base_model.eval()

print("Fetching Oracle config to match target modules...")
oracle_config = PeftConfig.from_pretrained(ORACLE_ADAPTER)
dummy_config = LoraConfig(target_modules=oracle_config.target_modules)

print("Adding dummy adapter...")
base_model.add_adapter(dummy_config, adapter_name="default")

print("Loading Oracle LoRA adapter...")
model = PeftModel.from_pretrained(
    base_model,
    ORACLE_ADAPTER,
    adapter_name="activation_oracle",
)
model.eval()

nonzero = sum(1 for n, p in model.named_parameters()
              if "activation_oracle" in n and "lora_B" in n
              and p.detach().float().norm() > 0)
assert nonzero > 0, "Oracle adapter did not load!"
print(f"Oracle successfully loaded! Nonzero matrices: {nonzero} | dtype: {model.dtype}")

INPUT_DEVICE = model.get_input_embeddings().weight.device
print("Input device:", INPUT_DEVICE)


## 5. Prompt format and residual-stream injection

**Verbatim from the reference notebook** — this is the logic you confirmed is correct. The source layer is only ever *quoted* in the prompt text (`get_introspection_prefix`); the oracle block that actually receives the injection is always `INJECTION_LAYER = 1`, fixed, regardless of which source layer the activation came from.

In [ ]:
SPECIAL_TOKEN = " ?"

def get_introspection_prefix(source_layer: int, num_positions: int) -> str:
    """Verbatim from the reference. The header and spacing are load-bearing."""
    return f"Layer: {source_layer}\n" + SPECIAL_TOKEN * num_positions + " \n"

def find_placeholders(token_ids, num_positions):
    """Locate the contiguous run of ' ?' placeholders in the tokenized prompt."""
    sid = tokenizer.encode(SPECIAL_TOKEN, add_special_tokens=False)[0]
    pos = [i for i, t in enumerate(token_ids) if t == sid][:num_positions]
    assert len(pos) == num_positions, f"found {len(pos)}/{num_positions} placeholders"
    return pos

def get_injection_submodule(layer: int):
    """The decoder block whose OUTPUT is modified. Always block 1, never the source layer."""
    for path in ("model.layers", "model.model.layers", "base_model.model.model.layers"):
        obj = model
        try:
            for part in path.split("."):
                obj = getattr(obj, part)
        except AttributeError:
            continue
        if isinstance(obj, torch.nn.ModuleList) and len(obj) > layer:
            return obj[layer]
    raise AttributeError("Could not locate the oracle's decoder layers.")

@contextlib.contextmanager
def temporary_hook(module, hook):
    h = module.register_forward_hook(hook)
    try:
        yield
    finally:
        h.remove()

def make_injection_hook(vectors, positions, coeff=1.0):
    """Paper Eq. 1:  h'_i = h_i + ||h_i|| * v_i / ||v_i||"""
    normed = [F.normalize(v, dim=-1).detach() for v in vectors]

    def hook_fn(module, _inp, output):
        if isinstance(output, tuple):
            resid, *rest = output; is_tuple = True
        else:
            resid, rest, is_tuple = output, [], False

        B, L, _ = resid.shape
        if L <= 1:
            return (resid, *rest) if is_tuple else resid

        for b in range(B):
            pos = torch.tensor(positions[b], dtype=torch.long, device=resid.device)
            if int(pos.max()) >= L:
                continue
            orig = resid[b, pos, :]
            inj = normed[b].to(resid.device) * orig.norm(dim=-1, keepdim=True) * coeff
            resid[b, pos, :] = orig + inj.to(resid.dtype).detach()

        return (resid, *rest) if is_tuple else resid
    return hook_fn

INJECTION_SUBMODULE = get_injection_submodule(INJECTION_LAYER)
print(f"Injection block configured at Layer {INJECTION_LAYER} (fixed, independent of source layer).")


## 6. `run_batch` — verbatim batching/generation logic

Only change from the reference: `LAYER` is now a parameter (it varies across the sweep) instead of a module-level constant.

In [ ]:
@torch.no_grad()
def run_batch(tasks, source_layer):
    """Verbatim batching logic from the reference notebook.
    `source_layer` replaces the module-level LAYER constant it used to close over."""
    seqs, positions, vectors = [], [], []
    for t in tasks:
        v = t["vectors"]; K = v.shape[0]
        prompt = get_introspection_prefix(source_layer, K) + t["question"]
        ids = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}], tokenize=True,
            add_generation_prompt=True, return_tensors=None,
            padding=False, enable_thinking=False,
        )
        seqs.append(ids)
        positions.append(find_placeholders(ids, K))
        vectors.append(v)

    maxlen, pad = max(len(s) for s in seqs), tokenizer.pad_token_id
    ids_b, attn_b, pos_b = [], [], []
    for ids, pos in zip(seqs, positions):
        p = maxlen - len(ids)
        ids_b.append([pad] * p + list(ids))
        attn_b.append([0] * p + [1] * len(ids))
        pos_b.append([x + p for x in pos])

    input_ids = torch.tensor(ids_b, dtype=torch.long, device=INPUT_DEVICE)
    attn = torch.tensor(attn_b, dtype=torch.long, device=INPUT_DEVICE)

    hook = make_injection_hook(vectors, pos_b, STEERING_COEFFICIENT)
    with temporary_hook(INJECTION_SUBMODULE, hook):
        out = model.generate(input_ids=input_ids, attention_mask=attn,
                             max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)

    answers = tokenizer.batch_decode(out[:, input_ids.shape[1]:], skip_special_tokens=True)

    return [{**{k: v for k, v in t.items() if k != "vectors"},
             "answer": a.strip()}
            for t, a in zip(tasks, answers)]

def run_all_batches(tasks, source_layer, label=""):
    """Chunk into BATCH_SIZE, print progress, return flat results list."""
    results = []
    n_batches = (len(tasks) + BATCH_SIZE - 1) // BATCH_SIZE
    for bi, i in enumerate(range(0, len(tasks), BATCH_SIZE), 1):
        chunk = tasks[i:i + BATCH_SIZE]
        results.extend(run_batch(chunk, source_layer))
        print(f"  [{label}] L{source_layer} batch {bi}/{n_batches} done ({len(results)}/{len(tasks)} generations)")
    return results

def save_json(obj, name):
    path = Path(OUTPUT_DIR) / name
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    print(f"Saved: {path}")
    return path


## 7. File loading helpers (doc / layer / status generic)

In [ ]:
def find_bundle(doc, status, layer, element):
    matches = list(Path(ACTIVATIONS_DIR).rglob(FILENAME_PATTERN.format(doc=doc, status=status, layer=layer, element=element)))
    return matches[0] if matches else None

def find_sidecar(doc, status, layer, element):
    matches = list(Path(ACTIVATIONS_DIR).rglob(SIDECAR_PATTERN.format(doc=doc, status=status, layer=layer, element=element)))
    return matches[0] if matches else None

def load_pt(path):
    return torch.load(path, map_location="cpu", weights_only=False)


## 8. A01–A08 — Task instruction probes

Same five questions and probe map as the reference notebook, now swept across every `(doc, layer)` pair in `DOC_IDS x LAYERS`, status `unsteered`.

In [ ]:
QUESTIONS_A01_08 = {
    "Q1": "What task is this activation associated with?",
    "Q2": "Which PICO element does this activation represent?",
    "Q3": "What semantic information is most strongly represented here?",
    "Q4": "What semantic concept or task does this activation represent?",
    "Q5": "What does this activation suggest the model is trying to do here?",
}

PROBE_MAP_A01_08 = {
    "p": ["A01", "A02"],
    "i": ["A03", "A04"],
    "c": ["A05", "A06"],
    "o": ["A07", "A08"],
}

def load_instruction_spans(doc, layer, status="unsteered"):
    """Instructions are identical across documents in the reference notebook's design,
    but we still key by (doc, layer) since files are sharded that way."""
    records = []
    for element in ELEMENTS:
        bundle_path = find_bundle(doc, status, layer, element)
        if bundle_path is None:
            print(f"  Warning: missing bundle for doc={doc} L{layer} {element}")
            continue

        payload = load_pt(bundle_path)
        acts = payload["activations"]
        text_meta = payload["metadata"].get("activation_text", {})

        for probe_id in PROBE_MAP_A01_08[element]:
            if probe_id in acts and acts[probe_id] is not None:
                vec = acts[probe_id].float()
                if vec.ndim == 1:
                    vec = vec.unsqueeze(0)
                records.append({
                    "doc": doc, "layer": layer, "element": element,
                    "probe_id": probe_id,
                    "span_text": text_meta.get(probe_id, "[UNKNOWN]"),
                    "vectors": vec,
                })
    return records

RESULTS_A01_08 = []
for layer in LAYERS:
    for doc in DOC_IDS:
        spans = load_instruction_spans(doc, layer)
        if not spans:
            continue
        tasks = [
            {"doc": doc, "layer": layer, "element": r["element"], "probe_id": r["probe_id"],
             "span_text": r["span_text"], "qid": qid, "question": q, "vectors": r["vectors"]}
            for r in spans for qid, q in QUESTIONS_A01_08.items()
        ]
        RESULTS_A01_08.extend(run_all_batches(tasks, layer, label=f"A01-08 doc={doc}"))

print(f"\nA01-08 total generations: {len(RESULTS_A01_08)}")


In [ ]:
# Clean grouped summary: A01-08
print("=" * 90)
print("ORACLE EXPLANATIONS: TASK INSTRUCTIONS (A01-A08)")
print("=" * 90)
grouped = defaultdict(list)
for r in RESULTS_A01_08:
    grouped[(r["doc"], r["layer"], r["probe_id"])].append(r)
for (doc, layer, probe_id), answers in list(grouped.items())[:6]:  # preview only; full data is in RESULTS_A01_08 / saved JSON
    sample = answers[0]
    print(f"\n[{sample['element'].upper()}] doc={doc} L{layer} | Probe {probe_id} | Text: {sample['span_text']!r}")
    print("-" * 90)
    for a in answers:
        print(f"Q: {a['question']}")
        print(f"A: {a['answer']}\n")
print(f"\n(...{max(0, len(grouped)-6)} more doc/layer/probe groups omitted from this preview; all are in the saved JSON)")

save_json(RESULTS_A01_08, "a01_08_instruction_probes.json")


## 9. A16 — Extracted PICO spans (full-span logic)

Same span-reconstruction logic as the reference notebook: A16 marks the extracted phrase, and `USE_FULL_SPAN` walks back to the A12 token range in the original abstract to pull the *full* multi-token span rather than a single collapsed vector.

In [ ]:
QUESTIONS_A16 = QUESTIONS_A01_08  # same five questions, per the reference notebook
USE_FULL_SPAN = True

def _abstract_offsets(abstract_ids):
    pieces = [tokenizer.decode([t]) for t in abstract_ids]
    offs, pos = [], 0
    for p in pieces:
        offs.append((pos, pos + len(p)))
        pos += len(p)
    return "".join(pieces), offs

def _span_token_range(text, offs, needle):
    i = text.find(needle.strip())
    if i == -1:
        return None
    j = i + len(needle.strip())
    idx = [t for t, (s, e) in enumerate(offs) if s < j and e > i]
    return (idx[0], idx[-1] + 1) if idx else None

def load_a16_spans(doc, layer, status="unsteered"):
    records = []
    for element in ELEMENTS:
        bundle_path = find_bundle(doc, status, layer, element)
        sidecar_path = find_sidecar(doc, status, layer, element)
        if bundle_path is None or sidecar_path is None:
            continue

        side = load_pt(sidecar_path)
        texts = side["metadata"]["activation_text"]["A16"]
        a16_single = side["activations"]["A16"]

        if USE_FULL_SPAN:
            bundle = load_pt(bundle_path)
            a12 = bundle["activations"]["A12"].float()
            meta = bundle["metadata"]
            s, e = meta["activation_token_spans"]["A12"]
            abstract_ids = meta["prompt_token_ids"][s:e]
            text, offs = _abstract_offsets(abstract_ids)

        for row, span_text in enumerate(texts):
            if USE_FULL_SPAN:
                rng = _span_token_range(text, offs, span_text)
                if rng is None or rng[1] > a12.shape[0]:
                    continue
                vec = a12[rng[0]:rng[1]].clone()
            else:
                vec = a16_single[row].float().unsqueeze(0)

            records.append({
                "doc": doc, "layer": layer, "element": element, "span_row": row,
                "span_text": span_text, "vectors": vec,
            })
    return records

RESULTS_A16 = []
for layer in LAYERS:
    for doc in DOC_IDS:
        spans = load_a16_spans(doc, layer)
        if not spans:
            continue
        tasks = [
            {"doc": doc, "layer": layer, "element": r["element"], "span_text": r["span_text"],
             "qid": qid, "question": q, "vectors": r["vectors"]}
            for r in spans for qid, q in QUESTIONS_A16.items()
        ]
        RESULTS_A16.extend(run_all_batches(tasks, layer, label=f"A16 doc={doc}"))

print(f"\nA16 total generations: {len(RESULTS_A16)}")


In [ ]:
print("=" * 90)
print("ORACLE EXPLANATIONS: EXTRACTED PICO SPANS (A16, full-span)")
print("=" * 90)
grouped = defaultdict(list)
for r in RESULTS_A16:
    grouped[(r["doc"], r["layer"], r["element"], r["span_text"])].append(r)
for (doc, layer, element, span_text), answers in list(grouped.items())[:6]:
    print(f"\n[{element.upper()}] doc={doc} L{layer} | Extracted: {span_text!r}")
    print("-" * 90)
    for a in answers:
        print(f"Q: {a['question']}")
        print(f"A: {a['answer']}\n")
print(f"\n(...{max(0, len(grouped)-6)} more groups omitted from this preview; all are in the saved JSON)")

save_json(RESULTS_A16, "a16_extracted_spans.json")


## 10. A09 — Pre-generation intent state

The token right before the model starts generating its answer (unsteered).

In [ ]:
QUESTIONS_A09 = {
    "Q1": "What task has the assistant been asked to perform?",
    "Q2": "What instructions is the assistant following?",
    "Q3": "What is the assistant about to do?",
    "Q4": "Can you describe what is happening in this text?",
    "Q5": "What are the PICO elements in the given abstract?",
}

def load_a09_tasks(doc, layer, status="unsteered"):
    tasks = []
    for element in ELEMENTS:
        bundle_path = find_bundle(doc, status, layer, element)
        if bundle_path is None:
            continue
        payload = load_pt(bundle_path)
        acts = payload["activations"]
        meta = payload["metadata"]
        if "A09" in acts and acts["A09"] is not None:
            token_text = meta.get("activation_text", {}).get("A09", "UNKNOWN")
            print(f"  [doc={doc} L{layer} {element.upper()}] A09 token: {token_text!r}")
            vec = acts["A09"].float()
            if vec.ndim == 1:
                vec = vec.unsqueeze(0)
            for qid, q in QUESTIONS_A09.items():
                tasks.append({"doc": doc, "layer": layer, "element": element,
                              "qid": qid, "question": q, "vectors": vec})
    return tasks

RESULTS_A09 = []
for layer in LAYERS:
    for doc in DOC_IDS:
        tasks = load_a09_tasks(doc, layer)
        if not tasks:
            continue
        RESULTS_A09.extend(run_all_batches(tasks, layer, label=f"A09 doc={doc}"))

print(f"\nA09 total generations: {len(RESULTS_A09)}")


In [ ]:
print("=" * 90)
print("ORACLE EXPLANATIONS: A09 (PRE-GENERATION INTENT STATE)")
print("=" * 90)
grouped = defaultdict(list)
for r in RESULTS_A09:
    grouped[(r["doc"], r["layer"], r["element"])].append(r)
for (doc, layer, element), answers in list(grouped.items())[:6]:
    print(f"\n[{element.upper()}] doc={doc} L{layer}")
    print("-" * 90)
    for a in answers:
        print(f"Q: {a['question']}")
        print(f"A: {a['answer']}\n")
print(f"\n(...{max(0, len(grouped)-6)} more groups omitted from this preview; all are in the saved JSON)")

save_json(RESULTS_A09, "a09_pregeneration_intent.json")


## 11. A13 — Steered assistant start token

Same token position as A09, but read from the `steered` activation dump.

In [ ]:
QUESTIONS_A13 = {
    "Q1": "What task is the assistant about to perform?",
    "Q2": "What instructions is the assistant following?",
    "Q3": "What is the assistant about to do?",
    "Q4": "What PICO element will the assistant extract?",
    "Q5": "Is the assistant about to output a JSON object?",
    "Q6": "What is the assistant about to write?",
}

def load_a13_tasks(doc, layer, status="steered"):
    tasks = []
    for element in ELEMENTS:
        bundle_path = find_bundle(doc, status, layer, element)
        if bundle_path is None:
            continue
        payload = load_pt(bundle_path)
        acts = payload["activations"]
        meta = payload["metadata"]
        if "A13" in acts and acts["A13"] is not None:
            # A13 is the steered version of A09, so it sits on the A09 token string
            token_text = meta.get("activation_text", {}).get("A09", "UNKNOWN")
            print(f"  [doc={doc} L{layer} {element.upper()}] A13 token: {token_text!r}")
            vec = acts["A13"].float()
            if vec.ndim == 1:
                vec = vec.unsqueeze(0)
            for qid, q in QUESTIONS_A13.items():
                tasks.append({"doc": doc, "layer": layer, "element": element,
                              "qid": qid, "question": q, "vectors": vec})
    return tasks

RESULTS_A13 = []
for layer in LAYERS:
    for doc in DOC_IDS:
        tasks = load_a13_tasks(doc, layer)
        if not tasks:
            continue
        RESULTS_A13.extend(run_all_batches(tasks, layer, label=f"A13 doc={doc}"))

print(f"\nA13 total generations: {len(RESULTS_A13)}")


In [ ]:
print("=" * 90)
print("ORACLE EXPLANATIONS: A13 (STEERED PRE-GENERATION INTENT)")
print("=" * 90)
grouped = defaultdict(list)
for r in RESULTS_A13:
    grouped[(r["doc"], r["layer"], r["element"])].append(r)
for (doc, layer, element), answers in list(grouped.items())[:6]:
    print(f"\n[{element.upper()}] doc={doc} L{layer}")
    print("-" * 90)
    for a in answers:
        print(f"Q: {a['question']}")
        print(f"A: {a['answer']}\n")
print(f"\n(...{max(0, len(grouped)-6)} more groups omitted from this preview; all are in the saved JSON)")

save_json(RESULTS_A13, "a13_steered_intent.json")


## 12. A15 — End-of-guidelines state

The very last token of the (unsteered) PICO guideline block.

In [ ]:
QUESTIONS_A15 = {
    "Q1": "What instructions is the assistant following?",
    "Q2": "What rules has the assistant been told to follow?",
    "Q3": "What PICO element do these instructions describe?",
    "Q4": "What task is the assistant about to perform?",
    "Q5": "Is this text a set of clinical guidelines?",
    "Q6": "What semantic concept is most strongly represented here?",
}

def load_a15_tasks(doc, layer, status="unsteered"):
    tasks = []
    for element in ELEMENTS:
        bundle_path = find_bundle(doc, status, layer, element)
        if bundle_path is None:
            continue
        payload = load_pt(bundle_path)
        acts = payload["activations"]
        meta = payload["metadata"]
        if "A15" in acts and acts["A15"] is not None:
            token_text = meta.get("activation_text", {}).get("A15", "UNKNOWN")
            print(f"  [doc={doc} L{layer} {element.upper()}] A15 token: {token_text!r}")
            vec = acts["A15"].float()
            if vec.ndim == 1:
                vec = vec.unsqueeze(0)
            for qid, q in QUESTIONS_A15.items():
                tasks.append({"doc": doc, "layer": layer, "element": element,
                              "qid": qid, "question": q, "vectors": vec})
    return tasks

RESULTS_A15 = []
for layer in LAYERS:
    for doc in DOC_IDS:
        tasks = load_a15_tasks(doc, layer)
        if not tasks:
            continue
        RESULTS_A15.extend(run_all_batches(tasks, layer, label=f"A15 doc={doc}"))

print(f"\nA15 total generations: {len(RESULTS_A15)}")


In [ ]:
print("=" * 90)
print("ORACLE EXPLANATIONS: A15 (END OF GUIDELINES STATE)")
print("=" * 90)
grouped = defaultdict(list)
for r in RESULTS_A15:
    grouped[(r["doc"], r["layer"], r["element"])].append(r)
if not grouped:
    print("WARNING: No A15 vectors found!")
else:
    for (doc, layer, element), answers in list(grouped.items())[:6]:
        print(f"\n[{element.upper()}] doc={doc} L{layer}")
        print("-" * 90)
        for a in answers:
            print(f"Q: {a['question']}")
            print(f"A: {a['answer']}\n")
    print(f"\n(...{max(0, len(grouped)-6)} more groups omitted from this preview; all are in the saved JSON)")

save_json(RESULTS_A15, "a15_end_of_guidelines.json")


## 13. Verify saved outputs

In [ ]:
for name in ["a01_08_instruction_probes.json", "a16_extracted_spans.json",
             "a09_pregeneration_intent.json", "a13_steered_intent.json",
             "a15_end_of_guidelines.json"]:
    path = Path(OUTPUT_DIR) / name
    exists = path.exists()
    n = None
    if exists:
        with open(path, "r", encoding="utf-8") as f:
            n = len(json.load(f))
    print(f"{name:35s} exists={exists!s:5s} records={n}")
